# Demonstration of `Molecule` creation

In this demo we create a `Molecule` of Methanol within `MDMC` through a script, and alternatively a `Molecule` of Paracetamol by using a `.cif` file. We also show how to apply bonds/forcefields to a `Molecule`.

In [ ]:
from MDMC.MD import Atom, Molecule, Bond, BondAngle, DihedralAngle, Universe, Dispersion, Shake, PPPM
from MDMC.gui import view
from MDMC.readers.configurations import read

Create the unique types of atoms in this molecule by creating the `C` and `O` atoms and one of the `H` atoms bonded to them each. The remaining `H` atoms will be added as copies of these after the bonds and bond-angles have been defined. The `name` property of the atom is used to map to the correct type of atom in the OPLSAA forcefield. See also the `Applying a ForceField` tutorial for more information on that step.

In [ ]:
# Define the unique atoms using the ForceField atom_type
# These can be seen in the oplsaa.dat file (MDMC/MD/force_fields/data/oplsaa.dat)
HC1 = Atom('H', position=[-0.7006,  0.3636,  0.8900], name='98', charge=0., atom_type=1)
C = Atom('C', position=[-0.3366, -0.1504,  0.0000], name='99', charge=0., atom_type=2)
O = Atom('O', position=[ 1.0849, -0.1713,  0.0000], name='96', charge=0., atom_type=3)
HO = Atom('H', position=[ 1.3606,  0.7699,  0.0000], name='97', charge=0., atom_type=4)

In [ ]:
# Create the bonds with harmonic potentials
CH_bond = Bond(C, HC1)
CO_bond = Bond(C, O, constrained=True)
OH_bond = Bond(O, HO)

In [ ]:
# Create the H-C-O and H-O-C bond angles
HCO_angle = BondAngle((HC1, C, O))
HOC_angle = BondAngle((HO, O, C))

# Create the H-C-O-H dihedral
HCOH_dihedral = DihedralAngle((HC1, C, O, HO))

# Duplicate the HC1 atom
HC2 = HC1.copy(position=[-0.7006,  0.3636, -0.8900])

# Create an HCH bond angle
HCH_angle = BondAngle((HC1, C, HC2))

# Duplicate the HC1 atom again
# This atom will have all bond (CH_bond) and bond angles (HCO_angle and HCH_angle) defined
HC3 = HC1.copy(position=[-0.7076, -1.1754,  0.0000])
atoms=[HC1, HC2, HC3, C, O, HO]

# Create the methanol Molecule
methanol = Molecule(atoms=atoms)

Visualise the Methanol molecule created.

In [ ]:
view(atoms)

Now create a `Universe` and add the methanol so we can add non-bonded interactions.

In [ ]:
universe = Universe(dimensions=15.0, constraint_algorithm=Shake(1e-5, 100), electrostatic_solver=PPPM(accuracy=1e-4))
universe.fill(methanol, num_density=0.01)
print(f'There are {universe.n_atoms} atoms in {universe.n_molecules} molecules in this universe')

Add dispersion interactions between the unique `atom_type`s. 

In [ ]:
HC_disp = Dispersion(universe, (1, 1), cutoff = 8.0, vdw_tail_correction=True)
C_disp = Dispersion(universe, (2, 2), cutoff = 8.0, vdw_tail_correction=True)
O_disp = Dispersion(universe, (3, 3), cutoff = 8.0, vdw_tail_correction=True)
HO_disp = Dispersion(universe, (4, 4), cutoff = 8.0, vdw_tail_correction=True)

Apply the `OPLSAA` forcefield to the universe.

In [ ]:

universe.add_force_field('OPLSAA')

Visualise the `Universe` and all the Methanol molcules inside it.

In [ ]:
view(universe)

# Alternative: create a Paracetamol `Molecule` from a `.cif` file

Specify the location of the file, read the file and create a `Molecule` object from the read-in data.

In [ ]:
paracetamol_path = '../../doc/tutorials/data/Paracetamol.cif'
atoms = read(paracetamol_path)
paracetamol = Molecule(atoms=atoms)

In [ ]:
view(paracetamol)

Investigation and validation methods for confirming that the `Molecule` was imported correctly

In [ ]:
# To see all of the Bond interactions, we can filter the paracetamol interactions by name
bonds = list(filter(lambda x: x.name == 'Bond', paracetamol.interactions))

# There are 20 bonds in total
print('Number of bonds: {}'.format(len(bonds)))

# We can cast the list of bonds to a set to see the number of unique bonds - in this case it is still 20
unique_bonds = set(bonds)
print('Number of unique bonds: {}'.format(len(unique_bonds)))

Optional: adding a `name` attribute to each atom imported from the `.cif` file. This can be used to apply a forcefield such as `OPLSAA` later. These `names` could be added after creation of the `Molecule` as well. See the `Applying a ForceField` tutorial for more info.

In [ ]:
atoms = read(paracetamol_path, names=['109', '177', # Oxygens
                                      '207', # Nitrogen
                                      '208', '108', '90', '178', '90', '90', '90', '185', # Carbons
                                      '85', '85', '85', '91', '91', '91', '91', '183', '110']) # Hydrogens
paracetamol = Molecule(atoms=atoms)

Optional: creating a `Universe` and adding a single `Molecule` of Paracetamol to it. 

In [ ]:
universe = Universe(10.)
universe.add_structure(paracetamol)
universe.add_force_field('OPLSAA')

In [ ]:
view(universe)